# Pré-processamento 5 — Triagem de concatenações

In [4]:
from pathlib import Path
import pandas as pd
from functools import lru_cache

arquivo = "Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx"
base = pd.read_excel(arquivo)
palavras = set(base["PALAVRA"].dropna().str.casefold())
pistas = {"não", "está", "você", "também", "então", "aqui", "agora",
          "meu", "minha", "para", "isso", "quero", "vamos", "tenho",
          "pode", "tem", "com", "sem", "uma", "ele", "ela"}

def separar(palavra):
    @lru_cache(None)
    def partes(inicio):
        if inicio == len(palavra):
            return [()]
        resultado = []
        for fim in range(inicio + 2, len(palavra) + 1):
            trecho = palavra[inicio:fim]
            if trecho in palavras and trecho != palavra:
                for resto in partes(fim):
                    if len(resto) < 3:
                        resultado.append((trecho, *resto))
        return resultado

    opcoes = partes(0)
    if opcoes:
        return min(opcoes, key=lambda p: (-sum(t in pistas for t in p), len(p), -min(map(len, p)), p))
    return ()

linhas = []
for palavra, lema, freq in base[["PALAVRA", "LEMA", "FREQ"]].itertuples(index=False, name=None):
    if not isinstance(palavra, str) or not palavra.isalpha() or len(palavra) < 5:
        continue
    partes = separar(palavra.casefold())
    fortes = [p for p in partes if p in pistas]
    repeticao = len(partes) > 1 and len(set(partes)) == 1
    if repeticao:
        prioridade, motivo = 0, "Repetição da mesma palavra"
    elif fortes:
        prioridade, motivo = 1, "Separação inclui " + ", ".join(fortes)
    elif partes and len(palavra) >= 8:
        prioridade, motivo = 2, "Separável em palavras da base"
    elif len(palavra) >= 12:
        prioridade, motivo = 3, "Comprimento de 12 letras ou mais"
    else:
        continue
    linhas.append((prioridade, palavra, lema, freq, " + ".join(partes) or "—", motivo))

triagem = pd.DataFrame(linhas, columns=["prioridade", "PALAVRA", "LEMA", "FREQ", "Separação", "Motivo"])
triagem = triagem.sort_values(["prioridade", "FREQ", "PALAVRA"], ascending=[True, False, True])
triagem = triagem.drop(columns="prioridade").reset_index(drop=True)
triagem.index += 1
print("Candidatos encontrados:", len(triagem))
saida = Path("Lista_triagem_concatenacoes.html")
tabela = triagem.rename_axis("Nº").to_html()
saida.write_text(
    '<!doctype html><html lang="pt-BR"><meta charset="utf-8">'
    '<title>Triagem de concatenações</title>'
    '<h1>Triagem de concatenações</h1>'
    f'<p>{len(triagem)} candidatos para revisão manual.</p>'
    + tabela + '</html>',
    encoding="utf-8",
)
print("Lista completa gerada:", saida.resolve())
display(triagem.head(20))

Candidatos encontrados: 334
Lista completa gerada: C:\Users\larin\Downloads\PRÉ PROCESSAMENTO V2.0\Lista_triagem_concatenacoes.html


,PALAVRA,LEMA,FREQ,Separação,Motivo
1,bombom,bombom,9,bom + bom,Repetição da mesma palavra
2,senão,senão,222,se + não,Separação inclui não
3,comeu,comer,200,com + eu,Separação inclui com
4,amarela,amanhã,57,amar + ela,Separação inclui ela
5,parado,parar,13,para + do,Separação inclui para
6,separa,separar,10,se + para,Separação inclui para
7,comemorando,comemorar,8,com + em + orando,Separação inclui com
8,poderosas,poderoso,7,pode + rosas,Separação inclui pode
9,comemorar,comemorar,5,com + em + orar,Separação inclui com
10,compras,compra,5,com + pras,Separação inclui com


In [2]:
# Exclusões aprovadas na revisão até a posição 200.
remover = {
    "xixixixi",
    "lambelambe",
    "tititi",
    "aiaiaiai",
    "batebate",
    "chuchuchuchu",
    "cocococo",
    "lalalalalalalalala",
    "agoraagora",
    "blindonblindon",
    "bubububu",
    "cliqueclique",
    "eieiei",
    "hahahahahaha",
    "lalalalalalalalalala",
    "maumau",
    "maçamaçamaça",
    "nananana",
    "nononono",
    "olhaolhaolha",
    "pegapega",
    "piupiupiupiu",
    "qualqualqual",
    "rodaroda",
    "tatatata",
    "temtem",
    "tiquetique",
    "tiquetiquetiquetique",
    "tudotudo",
    "aquicom",
    "aquideixa",
    "cabeçapode",
    "dexoveaqui",
    "essaaqui",
    "marcarcom",
    "olhaaqui",
    "parade",
    "paraeu",
    "paraonde",
    "paravoar",
    "vamosbrincar",
    "vamosfazer",
    "corderosa",
    "cocacola",
    "luluzona",
    "aiaiaiaiai",
    "brancadeneve",
    "grandepeixe",
    "pequenasereia",
    "piupiupiupiupiupiupiu",
    "tumtumtumtumtum",
    "peladona",
    "supernuvem",
    "muitobem",
    "babirussa",
    "caiofelipe",
    "casteloratimbum",
    "curative",
    "harryporter",
    "harrypotter",
    "papapapapapapapapapapa",
    "piupiupiupiupiu",
    "pontacabeça",
    "putzgrila",
}

# Decisões da rodada final.
correcoes = {
    "pulapula": "pula-pula",
    "guardachuva": "guarda-chuva",
    "arcoíris": "arco-íris",
    "peixeboi": "peixe-boi",
    "loboguará": "lobo guará",
    "viralata": "vira-lata",
    "beijaflor": "beija-flor",
    "mariafumaça": "maria-fumaça",
    "meiacalça": "meia-calça",
    "autofalante": "alto-falante",
    "estadosunidos": "estados unidos",
    "guardasol": "guarda-sol",
    "peixesapo": "peixe-sapo",
    "quadronegro": "quadro-negro",
    "quebracabeças": "quebra-cabeças",
    "bemvindo": "bem-vindo",
    "bichopreguiça": "bicho-preguiça",
    "dormimdo": "dormindo",
    "mariarita": "maria rita",
    "olhodevidro": "olho de vidro",
    "portamala": "porta-mala",
    "quartafeira": "quarta-feira",
    "sextafeira": "sexta-feira",
    "riodejaneiro": "Rio de Janeiro",
    "sombrancelha": "sobrancelha",
    "sombrancelhas": "sobrancelhas",
    "divagarzinho": "devagarzinho",
    "estreladomar": "estrela-do-mar",
    "mariacarlota": "maria carlota",
    "perigosesima": "perigosíssima",
    "reencontrálos": "reencontrá-los",
    "papainoel": "papai-noel"
}

remover.update({
    "ticotico",
    "piuípiuípiuí",
    "brincaar",
    "grilofalante",
    "papapapapapapapapapapa",
    "simsalabimsalabimbimbim",
    "superlegal",
    "sobrarem",
    "thcitchitchitchi",
    "amelieolha",
    "amelievai",
    "bepepilantra",
    "boniestou",
    "brincado",
    "brincarcomigo",
    "brincarde",
    "cantandobrincando",
    "carademau",
    "corderosas",
    "cuidocuidado",
    "deixaeuver",
    "dinotrem",
    "donadecasa",
    "dovagabundo",
    "esperoolha",
    "estadounidos",
    "estoufazendo",
    "fazeroutro",
    "ficarvendendo",
    "fumaçaolha",
    "gramamas",
    "guardaar",
    "jogolonge",
    "lalalalala",
    "lambedor",
    "lobomaudesmaiou",
    "luizasette",
    "luizausa",
    "macacula",
    "mantanosseratope",
    "maradona",
    "marchasoldado",
    "meninolobo",
    "montarsua",
    "motherinvestigator",
    "muitogrande",
    "muitoperto",
    "nossocasamento",
    "pegaesconde",
    "pernadepau",
    "peticolecola",
    "pompompompompompompom",
    "portacaneta",
    "princesaaurora",
    "príncipefelipe",
    "queconversar",
    "quercomer",
    "quintaanimada",
    "semigente",
    "superalto",
    "superbonito",
    "superdiferente",
    "supergostoso",
    "superhomem",
    "supersol",
    "tatatatata",
    "tatatatatata",
    "terradonunca",
    "tomaraquecaia",
    "tomouseu",
    "viscondedesabugosa",
    "xixiiiiii",
    "arquiospiteriquix",
    "pterodactilo",
    "ciamortirano",
    "orisquidocado",
    "participants",
    "piperanodonte",
    "pteranodonte",
    "adamaeovagabundo",
    "backyardigan",
    "mantanossera",
    "orroorroorro",
    "participantschi",
    "sinsalabinsalabinbinbin",
    "tchitchitchi",
    "tchuquitchuquitchuqui",
    "tchutchutchu",
    "tricerapodes",
    "asferiasdabruxonilda",
    "atartarugaquequeriavoar",
    "backyardgans",
    "balangandans",
    "beelelenocirco",
    "bigtreinador",
    "bleubleubleu",
    "blinblonblonblin",
    "blublublublu",
    "bonecojoaquim",
    "brumbrumbrumbrum",
    "bububububububu",
    "bíbliadosanimais",
    "chichichichichi",
    "comocontarcrocodilos",
    "conversandos",
    "correcutianacasadatia",
    "creamcracker",
    "cumpriiiiido",
    "cutecutecute",
    "dadadadadadodododo",
    "desdetrodonte",
    "destrambelha",
    "espaçodavila",
    "esprimentando",
    "fiiiiiiuuuuuu",
    "gragrandmother",
    "grandparents",
    "greatgrandmother",
    "hermanotauro",
    "huuuuuuummmm",
    "iiiiiiiiiiriiioo",
    "iiiiiiiiiriii",
    "ilhadosprazeres",
    "interviewers",
    "machadodeassis",
    "magneticgiro",
    "manobrasolha",
    "nananimnanam",
    "ocoelhoquincaseopotedeouro",
    "ornitominodois",
    "outonoinverno",
    "papapapapapapa",
    "parafernalhona",
    "perigosesimo",
    "piriquitisse",
    "piuiticticpiui",
    "ponponponponpon",
    "protosseratope",
    "ptveronesichi",
    "quebrascabeças",
    "queinmqueinm",
    "salabimbimbim",
    "sallyscouter",
    "sobrehumanas",
    "tchatchatchatchan",
    "tchetchetche",
    "tchiatchiatchia",
    "tchiiiiitchiiii",
    "tchithcithci",
    "tchithcithcitchi",
    "tchucotchucotchucotchuco",
    "tchucotchucothuco",
    "tchutchururururuuuuuuuuuu",
    "tesselopatato",
    "tititititititi",
    "tralalalalala",
    "trisserapodes",
    "trisseraptor",
    "tucutucutucu",
    "tulutulutulu",
    "turururururu",
    "tururutururutururuprupru",
    "umacoroaparaguilherme",
    "meudeus",
})


In [3]:
# Execute depois da célula que define a lista remover.
import pandas as pd

arquivo = "Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx"
base_atual = pd.read_excel(arquivo)
mascara_remover = base_atual["PALAVRA"].isin(remover)
encontradas = base_atual.loc[mascara_remover, "PALAVRA"].value_counts()
resultado = base_atual.loc[~mascara_remover].copy()

print("=== CORREÇÕES ENCONTRADAS ===")
total_corrigido = 0
for coluna in ["PALAVRA", "LEMA"]:
    for origem, destino in correcoes.items():
        quantidade = int(resultado[coluna].eq(origem).sum())
        if quantidade:
            print(f"{coluna}: {origem} -> {destino} ({quantidade})")
            total_corrigido += quantidade
    resultado[coluna] = resultado[coluna].replace(correcoes)

print("Repetições em PALAVRA após as alterações:", int(resultado["PALAVRA"].duplicated().sum()))

print("=== PALAVRAS QUE SERÃO REMOVIDAS ===")
for palavra, quantidade in encontradas.items():
    print(f"{palavra}: {quantidade}")
print("Registros que serão removidos:", int(mascara_remover.sum()))

if mascara_remover.any() or total_corrigido:
    resultado.to_excel(arquivo, index=False)
    print("Arquivo atualizado:", arquivo)
else:
    print("Nenhuma alteração pendente.")

print("Total de registros restantes:", len(resultado))
print("Repetições restantes em PALAVRA:", int(resultado["PALAVRA"].duplicated().sum()))


=== CORREÇÕES ENCONTRADAS ===
PALAVRA: pulapula -> pula-pula (1)
PALAVRA: guardachuva -> guarda-chuva (1)
PALAVRA: arcoíris -> arco-íris (1)
PALAVRA: peixeboi -> peixe-boi (1)
PALAVRA: loboguará -> lobo guará (1)
PALAVRA: viralata -> vira-lata (1)
PALAVRA: beijaflor -> beija-flor (1)
PALAVRA: mariafumaça -> maria-fumaça (1)
PALAVRA: meiacalça -> meia-calça (1)
PALAVRA: autofalante -> alto-falante (1)
PALAVRA: estadosunidos -> estados unidos (1)
PALAVRA: guardasol -> guarda-sol (1)
PALAVRA: peixesapo -> peixe-sapo (1)
PALAVRA: quadronegro -> quadro-negro (1)
PALAVRA: quebracabeças -> quebra-cabeças (1)
PALAVRA: bemvindo -> bem-vindo (1)
PALAVRA: bichopreguiça -> bicho-preguiça (1)
PALAVRA: dormimdo -> dormindo (1)
PALAVRA: mariarita -> maria rita (1)
PALAVRA: olhodevidro -> olho de vidro (1)
PALAVRA: portamala -> porta-mala (1)
PALAVRA: quartafeira -> quarta-feira (1)
PALAVRA: sextafeira -> sexta-feira (1)
PALAVRA: riodejaneiro -> Rio de Janeiro (1)
PALAVRA: sombrancelha -> sobrancelha 